In [7]:
import pandas as pd
calendier=pd.read_csv("calendar.csv")
print(calendier.head())
print(f" Les differents colonnes:\n{calendier.columns}")
print(f"{calendier['date']}")
print(calendier['price'].isna().sum())
print(f"les colonnes avec des valeurs nulles:{calendier.isna().sum()}")
##Transformer le type de le date en datetime
calendier['date']=pd.to_datetime(calendier['date'])
print(calendier['date'])
##Calculons le taux d'occupation dans la colonne avalaible
print(calendier['available'].value_counts())
calendier['available_norm']=calendier['available'].map({'f':1,'t':0})
print(f"l'occupation des apparts {calendier['available_norm'].value_counts()}")
#app_loger=calendier[calendier['available_norm']==0].mean()
print(f"La moyenne des appartements loger sont :")
#app_pas_loger=calendier[calendier['available_norm']==1].mean()
print(f"La moyenne des appartements loger sont :")
calendier['month']=calendier['date'].dt.month
print(f"les mois extraits sont:\n{calendier['month']}")
taux_occupation=calendier.groupby(['month','listing_id'])['available_norm'].mean().reset_index()
print(f"les logements par mois:{taux_occupation.value_counts()}")
taux_occupation_annuel = calendier.groupby('listing_id')['available_norm'].mean().reset_index()
print(f"le taux d'occupation annuel:\n{taux_occupation_annuel['available_norm']}")
print(f"la date min et max :\n{calendier.groupby('year')['date'].agg(['min', 'max'])}")
print(calendier['maximum_nights'].value_counts())
print(calendier['minimum_nights'].value_counts())
print(calendier[['maximum_nights','minimum_nights']].describe())
print(f"maximum_nuits:{calendier['maximum_nights'].median()}")
print(f"minimum_nuits:{calendier['minimum_nights'].median()}")



   listing_id        date available  price  adjusted_price  minimum_nights  \
0    12674667  2025-09-18         f    NaN             NaN               1   
1    12674667  2025-09-19         t    NaN             NaN               1   
2    12674667  2025-09-20         t    NaN             NaN               1   
3    12674667  2025-09-21         t    NaN             NaN               1   
4    12674667  2025-09-22         f    NaN             NaN               1   

   maximum_nights  
0               8  
1               8  
2               8  
3               8  
4               8  
 Les differents colonnes:
Index(['listing_id', 'date', 'available', 'price', 'adjusted_price',
       'minimum_nights', 'maximum_nights'],
      dtype='object')
0          2025-09-18
1          2025-09-19
2          2025-09-20
3          2025-09-21
4          2025-09-22
              ...    
4471975    2026-09-13
4471976    2026-09-14
4471977    2026-09-15
4471978    2026-09-16
4471979    2026-09-17
Name: da

KeyError: 'year'

In [15]:

ann_app=pd.read_csv('listings.csv')
print(ann_app.head())
print(ann_app.columns)
print(f"maximum_nights:\n{ann_app['maximum_nights'].describe()}")
print(f"minimum_nights:{ann_app['minimum_nights'].describe()}")
print(ann_app['maximum_nights'].median())
print(ann_app['minimum_nights'].median())
print(ann_app['maximum_nights'].value_counts())
print(ann_app['minimum_nights'].value_counts())
#On n'a pas besoin des logements logues on a seulement besoin de courtes durées pour les votageurs
listings_propre = ann_app[ann_app['minimum_nights'] <= 30].copy()
print(listings_propre.head())
print(listings_propre.columns)
listings_propre2=listings_propre[["id","neighbourhood_cleansed","room_type","accommodates","bedrooms","price", "minimum_nights","maximum_nights","amenities","review_scores_rating","estimated_occupancy_l365d"]].copy()
print(listings_propre2.columns)
print(listings_propre2['price'].head())
print(listings_propre2['price'].value_counts())
listings_propre2['price']=listings_propre2['price'].str.replace('$', '',regex=False)
listings_propre2['price']=listings_propre2['price'].str.replace(',', '',regex=False)
listings_propre2['price']=listings_propre2['price'].str.strip()
listings_propre2['price']=listings_propre2['price'].astype(float)
print(listings_propre2['price'].head())
#Attaquons la colonnes equipements
print(listings_propre2['amenities'][0])
#Extraire les lignes ou le mots wifi est repere puis mettons le dans une colonne
listings_propre2['Wifi']=listings_propre2['amenities'].str.lower().str.contains('wifi').astype(int)
print(f"les maisons qui detient du wifi:\n{listings_propre2['Wifi'].value_counts()}")
#Les maisons qui ont de la cuisine
listings_propre2['kitchen']=listings_propre2['amenities'].str.lower().str.contains('kitchen').astype(int)
print(f"les maisons qui ont de la cuisine:\n{listings_propre2['kitchen'].value_counts()}")
#Les maisons qui ont la machine à laver:
# Version sécurisée qui élimine les lave-vaisselles
listings_propre2['has_washer'] = (
    listings_propre2['amenities'].str.lower()
    .str.contains(r'(?<!dish)washer', regex=True, na=False)
).astype(int)
print(f"les maisons qui ont la machine à laver:\n{listings_propre2['has_washer'].value_counts()}")
#Les maisons qui ont de la climatisation:
listings_propre2['air conditioning']=listings_propre2['amenities'].str.lower().str.contains('air conditioning').astype(int)
print(f"les maisons qui detiennent des climatiseurs:\n{listings_propre2['air conditioning'].value_counts()}")
import ast
from collections import Counter
# on transforme le texte en vraie liste, puis on compte chaque équipement
tous = listings_propre2['amenities'].dropna().apply(ast.literal_eval)
compteur = Counter(equip for liste in tous for equip in liste)
def parking_gratuit_sur_place(liste):
    for equip in liste:              # on prend chaque équipement, un par un
        e = equip.lower()            # tout en minuscules ("Free" → "free")
        est_parking = ('parking' in e) or ('garage' in e) or ('driveway' in e)
        if est_parking and 'free' in e and 'on premises' in e:
            return 1                 # dès qu'UN équipement coche les 3 → c'est oui, on s'arrête
    return 0                         # si on a tout vu sans rien trouver → c'est non
listes = listings_propre2['amenities'].apply(ast.literal_eval)  # convertit le texte en liste, pour tous
listings_propre2['parking_gratuit'] = listes.apply(parking_gratuit_sur_place)  # 0 ou 1, pour tous
print(f"les maisons qui ont un parking gratuit:\n{listings_propre2['parking_gratuit'].value_counts()}")
print(listings_propre2['estimated_occupancy_l365d'].head())


       id                          listing_url       scrape_id last_scraped  \
0  222887  https://www.airbnb.com/rooms/222887  20250918041729   2025-09-18   
1  317273  https://www.airbnb.com/rooms/317273  20250918041729   2025-09-18   
2  317658  https://www.airbnb.com/rooms/317658  20250918041729   2025-09-18   
3  333031  https://www.airbnb.com/rooms/333031  20250918041729   2025-09-18   
4  365993  https://www.airbnb.com/rooms/365993  20250918041729   2025-09-18   

        source                                               name  \
0  city scrape           Spectacular view, full air-con, elevator   
1  city scrape       Luxury, spacious, patio, near public gardens   
2  city scrape  Key to Bordeaux · fairytale view, 2 bd + elevator   
3  city scrape      STUDIO BORDEAUX TRIANGLE D OR ***** Climatisé   
4  city scrape                   Modern&comfortable 3 rooms apart   

                                         description  \
0  Imagine yourself relaxing on a 12m² private te...  

In [ ]:
import numpy as np
import ast

# ══════════════════════════════════════════════════════════════
# 1. FUSION  listings × taux d'occupation
# ══════════════════════════════════════════════════════════════
master = listings_propre2.merge(
    taux_occupation_annuel, left_on='id', right_on='listing_id', how='left'
).drop(columns=['listing_id']).rename(columns={'available_norm': 'taux_occupation'})

# Supprimer les colonnes d'équipements créées manuellement (remplacées par l'approche dynamique)
cols_manuels = ['Wifi', 'kitchen', 'has_washer', 'air conditioning', 'parking_gratuit']
master = master.drop(columns=[c for c in cols_manuels if c in master.columns])

# ══════════════════════════════════════════════════════════════
# 2. TERCILES DE PRIX  par quartier + type de logement
# ══════════════════════════════════════════════════════════════
def safe_qcut(x):
    try:
        return pd.qcut(x, q=3, labels=['bas', 'moyen', 'haut'], duplicates='drop')
    except ValueError:
        return pd.Series('unique', index=x.index)

master['tranche_prix'] = master.groupby(
    ['room_type', 'neighbourhood_cleansed']
)['price'].transform(safe_qcut)

print(f"Répartition des tranches de prix :\n{master['tranche_prix'].value_counts()}\n")

# ══════════════════════════════════════════════════════════════
# 3. MATRICE BINAIRE DYNAMIQUE  de TOUS les équipements
# ══════════════════════════════════════════════════════════════
amenities_lists = master['amenities'].apply(ast.literal_eval)
amenities_lists = amenities_lists.apply(lambda lst: [a.strip() for a in lst])

amenities_df = pd.DataFrame(
    [{a: 1 for a in lst} for lst in amenities_lists],
    index=master.index
).fillna(0).astype(int)

# Filtre anti-bruit : garder uniquement les équipements présents dans ≥ 2% des annonces
freq = amenities_df.mean()
equip_cols = freq[freq >= 0.02].index.tolist()
amenities_df = amenities_df[equip_cols]

print(f"Nombre d'équipements retenus (≥2% de présence) : {len(equip_cols)}")
print(f"\nTop 20 équipements les plus fréquents :")
print(freq[equip_cols].sort_values(ascending=False).head(20))

# ══════════════════════════════════════════════════════════════
# 4. CLASSIFICATION  top / flop  par segment
# ══════════════════════════════════════════════════════════════
group_cols = ['room_type', 'neighbourhood_cleansed', 'tranche_prix']

master['seuil'] = master.groupby(group_cols)['estimated_occupancy_l365d'].transform(
    lambda x: x.quantile(0.75)
)
master['classement'] = np.where(master['estimated_occupancy_l365d'] >= master['seuil'], 1, 0)

master['taille_groupe'] = master.groupby(group_cols)['id'].transform('count')
master['classement'] = np.where(master['taille_groupe'] <= 10, -1, master['classement'])

print(f"\nRépartition des classements :\n{master['classement'].value_counts()}")

# ══════════════════════════════════════════════════════════════
# 5. PROFIL DYNAMIQUE DES TOPS  (équipements + nuits minimum)
# ══════════════════════════════════════════════════════════════
amenities_analysis = amenities_df.copy()
for col in group_cols + ['classement', 'minimum_nights']:
    amenities_analysis[col] = master[col].values

top_data = amenities_analysis[amenities_analysis['classement'] == 1]
profil_equip_top = top_data.groupby(group_cols)[equip_cols].mean()
profil_nuits_top = top_data.groupby(group_cols)['minimum_nights'].median()

print(f"\nExemple de profil top (premier groupe) :")
print(profil_equip_top.iloc[0].sort_values(ascending=False).head(10))

# ══════════════════════════════════════════════════════════════
# 6. RECOMMANDATIONS DYNAMIQUES
# ══════════════════════════════════════════════════════════════
SEUIL_EQUIP = 0.50   # l'équipement doit être chez ≥50% des tops pour être recommandé
TOP_N = 5            # maximum 5 recommandations d'équipements par logement

def generer_recommandations(idx):
    row_master = master.loc[idx]
    row_equip = amenities_df.loc[idx]

    if row_master['classement'] == -1:
        return "Pas assez de logements comparables dans votre segment pour un diagnostic fiable."
    if row_master['classement'] == 1:
        return "Votre logement fait déjà partie des meilleurs de votre segment. Continuez ainsi !"

    key = tuple(row_master[c] for c in group_cols)
    if key not in profil_equip_top.index:
        return "Pas de profil de référence pour votre segment."

    top_profile = profil_equip_top.loc[key]

    # Trouver les équipements manquants que les tops possèdent, triés par importance
    gaps = []
    for equip in equip_cols:
        top_rate = top_profile[equip]
        if row_equip[equip] == 0 and top_rate >= SEUIL_EQUIP:
            gaps.append((equip, top_rate))

    gaps.sort(key=lambda x: x[1], reverse=True)

    recos = []
    for equip, rate in gaps[:TOP_N]:
        pct = int(rate * 100)
        recos.append(f"Ajouter « {equip} » (présent chez {pct}% des tops de votre segment)")

    # Recommandation nuits minimum
    if key in profil_nuits_top.index:
        med_top = profil_nuits_top.loc[key]
        if row_master['minimum_nights'] >= med_top + 2:
            recos.append(
                f"Réduire vos nuits minimum à {int(med_top)} (les tops acceptent des séjours plus courts)"
            )

    if not recos:
        return "Votre logement correspond déjà au profil des meilleurs de votre segment."

    return "\n".join(f"{i}. {r}" for i, r in enumerate(recos, 1))

master['recommandation_finale'] = [generer_recommandations(idx) for idx in master.index]

print(f"\nDistribution des recommandations :")
print(master['recommandation_finale'].value_counts().head(10))

# ══════════════════════════════════════════════════════════════
# 7. EXPORT
# ══════════════════════════════════════════════════════════════
master.to_csv("master_final.csv", index=False)
print(f"\nFichier master_final.csv exporté ({master.shape[0]} lignes, {master.shape[1]} colonnes)")

# ══════════════════════════════════════════════════════════════
# 8. TEST : logements avec plusieurs recommandations
# ══════════════════════════════════════════════════════════════
flops = master[master['classement'] == 0]
multi_recos = flops[flops['recommandation_finale'].str.count('\n') >= 2]
print(f"\n=== LOGEMENTS DE TEST POUR STREAMLIT ===")
print(f"Nombre de logements avec 3+ recommandations : {len(multi_recos)}")
if len(multi_recos) > 0:
    print(f"Exemples d'IDs : {multi_recos['id'].head(10).tolist()}")